In [0]:
# 01 - Importar bibliotecas e carregar as tabelas necessárias

from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)
from pyspark.ml.regression import LinearRegression

df_features = spark.table(
    "workspace.projeto_integrador.gold_features_ml"
)

df_indicadores = spark.table(
    "workspace.projeto_integrador.gold_indicadores_municipais"
)

print("Registros da base de features:", df_features.count())
print("Registros da Gold de indicadores:", df_indicadores.count())

In [0]:
# 02 - Identificar os municípios participantes do modelo

municipios_modelo = (
    df_features
    .select("codigo_ibge")
    .distinct()
)

print(
    "Quantidade de municípios:",
    municipios_modelo.count()
)

In [0]:
# 03 - Recuperar histórico recente para construir as features de 2026

historico_2023_2025 = (
    df_indicadores
    .join(
        municipios_modelo,
        on="codigo_ibge",
        how="inner"
    )
    .filter(
        F.col("ano").isin(2023, 2024, 2025)
    )
)

display(
    historico_2023_2025
    .orderBy("codigo_ibge", "ano")
    .limit(30)
)

In [0]:
# 04 - Criar estrutura de dados para previsão de 2026

base_2026 = (
    historico_2023_2025
    .groupBy(
        "codigo_ibge",
        "municipio",
        "uf",
        "estado",
        "regiao_brasil"
    )
    .agg(
        F.max(
            F.when(
                F.col("ano") == 2025,
                F.col("valor_indicador_municipio")
            )
        ).alias("valor_ano_anterior"),

        F.max(
            F.when(
                F.col("ano") == 2024,
                F.col("valor_indicador_municipio")
            )
        ).alias("valor_2_anos_antes"),

        F.max(
            F.when(
                F.col("ano") == 2023,
                F.col("valor_indicador_municipio")
            )
        ).alias("valor_3_anos_antes")
    )

    .withColumn(
        "ano",
        F.lit(2026)
    )
)

display(
    base_2026
    .orderBy("codigo_ibge")
    .limit(20)
)

In [0]:
# 05 - Criar features históricas para previsão de 2026

base_2026 = (
    base_2026

    .withColumn(
        "media_3_anos_anteriores",
        (
            F.col("valor_ano_anterior") +
            F.col("valor_2_anos_antes") +
            F.col("valor_3_anos_antes")
        ) / 3
    )

    .withColumn(
        "diferenca_ultimo_ano",
        F.col("valor_ano_anterior") -
        F.col("valor_2_anos_antes")
    )

    .withColumn(
        "variacao_percentual_historica",
        F.when(
            F.col("valor_2_anos_antes") > 0,
            (
                (
                    F.col("valor_ano_anterior") -
                    F.col("valor_2_anos_antes")
                )
                / F.col("valor_2_anos_antes")
            ) * 100
        ).otherwise(0.0)
    )

    .withColumn(
        "razao_ultimo_media_3_anos",
        F.when(
            F.col("media_3_anos_anteriores") > 0,
            F.col("valor_ano_anterior") /
            F.col("media_3_anos_anteriores")
        ).otherwise(0.0)
    )
)

display(
    base_2026
    .orderBy("codigo_ibge")
    .limit(20)
)

In [0]:
# 06 - Validar a base preparada para 2026

print(
    "Registros preparados para previsão:",
    base_2026.count()
)

print(
    "Municípios:",
    base_2026
    .select("codigo_ibge")
    .distinct()
    .count()
)

nulos_2026 = base_2026.select(
    F.sum(
        F.when(
            F.col("valor_ano_anterior").isNull(),
            1
        ).otherwise(0)
    ).alias("nulos_2025"),

    F.sum(
        F.when(
            F.col("valor_2_anos_antes").isNull(),
            1
        ).otherwise(0)
    ).alias("nulos_2024"),

    F.sum(
        F.when(
            F.col("valor_3_anos_antes").isNull(),
            1
        ).otherwise(0)
    ).alias("nulos_2023")
)

display(nulos_2026)

In [0]:
# 07 - Treinar o modelo definitivo utilizando todo o histórico disponível

df_treino_final = df_features.filter(
    F.col("ano") <= 2025
)

indexador_uf = StringIndexer(
    inputCol="uf",
    outputCol="uf_index",
    handleInvalid="keep"
)

indexador_regiao = StringIndexer(
    inputCol="regiao_brasil",
    outputCol="regiao_index",
    handleInvalid="keep"
)

encoder = OneHotEncoder(
    inputCols=[
        "uf_index",
        "regiao_index"
    ],
    outputCols=[
        "uf_ohe",
        "regiao_ohe"
    ]
)

features_modelo = [
    "valor_ano_anterior",
    "valor_2_anos_antes",
    "valor_3_anos_antes",
    "media_3_anos_anteriores",
    "diferenca_ultimo_ano",
    "variacao_percentual_historica",
    "razao_ultimo_media_3_anos",
    "uf_ohe",
    "regiao_ohe"
]

assembler = VectorAssembler(
    inputCols=features_modelo,
    outputCol="features"
)

regressao_linear = LinearRegression(
    featuresCol="features",
    labelCol="target",
    predictionCol="prediction_ml",
    maxIter=100,
    regParam=0.1,
    elasticNetParam=0.0
)

pipeline_final = Pipeline(
    stages=[
        indexador_uf,
        indexador_regiao,
        encoder,
        assembler,
        regressao_linear
    ]
)

modelo_2026 = pipeline_final.fit(
    df_treino_final
)

print(
    "Modelo definitivo treinado com",
    df_treino_final.count(),
    "registros."
)

In [0]:
# 08 - Gerar previsões para 2026

previsoes_2026 = (
    modelo_2026
    .transform(base_2026)

    .withColumn(
        "prediction_ml",
        F.greatest(
            F.col("prediction_ml"),
            F.lit(0.0)
        )
    )

    # Baseline: repetir o valor de 2025
    .withColumn(
        "prediction_baseline",
        F.col("valor_ano_anterior")
    )

    # Valor inteiro mais adequado à interpretação de quantidade de exames
    .withColumn(
        "previsao_ml",
        F.round(
            F.col("prediction_ml")
        ).cast("long")
    )

    .withColumn(
        "previsao_baseline",
        F.round(
            F.col("prediction_baseline")
        ).cast("long")
    )
)

display(
    previsoes_2026
    .select(
        "codigo_ibge",
        "municipio",
        "uf",
        "ano",
        "valor_ano_anterior",
        "previsao_ml",
        "previsao_baseline"
    )
    .orderBy(
        F.desc("previsao_ml")
    )
    .limit(30)
)

In [0]:
# 09 - Calcular variação prevista para 2026

resultado_2026 = (
    previsoes_2026

    .withColumn(
        "variacao_prevista_ml_percentual",
        F.when(
            F.col("valor_ano_anterior") > 0,
            F.round(
                (
                    (
                        F.col("previsao_ml") -
                        F.col("valor_ano_anterior")
                    )
                    / F.col("valor_ano_anterior")
                ) * 100,
                2
            )
        )
    )

    .select(
        "codigo_ibge",
        "municipio",
        "uf",
        "estado",
        "regiao_brasil",
        "ano",

        F.col(
            "valor_ano_anterior"
        ).alias("valor_2025"),

        "previsao_ml",
        "previsao_baseline",
        "variacao_prevista_ml_percentual"
    )
)

display(
    resultado_2026
    .orderBy(
        F.desc("previsao_ml")
    )
    .limit(30)
)

In [0]:
# 10 - Resumo das previsões para 2026

resumo_2026 = (
    resultado_2026
    .agg(
        F.count("*").alias("municipios"),

        F.sum(
            "valor_2025"
        ).alias("total_2025"),

        F.sum(
            "previsao_ml"
        ).alias("total_previsto_ml_2026"),

        F.sum(
            "previsao_baseline"
        ).alias("total_previsto_baseline_2026")
    )

    .withColumn(
        "variacao_total_ml_percentual",
        F.round(
            (
                (
                    F.col("total_previsto_ml_2026") -
                    F.col("total_2025")
                )
                / F.col("total_2025")
            ) * 100,
            2
        )
    )
)

display(resumo_2026)

In [0]:
# 11 - Salvar previsões de 2026 no Unity Catalog

resultado_2026.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.projeto_integrador.gold_previsoes_2026"
    )

print(
    "Tabela gold_previsoes_2026 salva com sucesso."
)

In [0]:
# 12 - Validar a tabela final de previsões

gold_previsoes_2026 = spark.table(
    "workspace.projeto_integrador.gold_previsoes_2026"
)

print(
    "Quantidade de previsões:",
    gold_previsoes_2026.count()
)

print(
    "Quantidade de municípios:",
    gold_previsoes_2026
    .select("codigo_ibge")
    .distinct()
    .count()
)

display(
    gold_previsoes_2026
    .orderBy(
        F.desc("previsao_ml")
    )
    .limit(20)
)